# SCE-Net One-Class (Deep SVDD) с максимальным переиспользованием baseline

Этот ноутбук переиспользует логику из `sce_net_fashion_compatibility_pair_classification.ipynb`:
- CLIP backbone (`openai/clip-vit-base-patch32`),
- конфиг в стиле dataclass,
- RAM-cache изображений,
- Dataset/Collate и DataLoader-пайплайн,
- архитектуру SCE-Net (condition masks + weight branch).

Изменение только в целевой функции: вместо pair BCE используем Deep SVDD objective для обучения на `y=1` парах.


In [ ]:
import random
import time
from dataclasses import dataclass
from typing import Dict

import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import CLIPModel, CLIPProcessor
from sklearn.metrics import precision_recall_curve, f1_score, roc_auc_score, average_precision_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)


In [ ]:
@dataclass
class OneClassConfig:
    hf_model_name: str = 'patrickjohncyh/fashion-clip'

    # SCE-Net
    num_conditions: int = 5
    condition_hidden: int = 1024
    dropout: float = 0.1

    # Deep SVDD head
    svdd_dim: int = 256

    # optimization
    batch_size: int = 64
    lr: float = 1e-4
    weight_decay: float = 1e-4
    epochs: int = 12
    use_amp: bool = True
    freeze_backbone: bool = False
    clip_grad_norm: float = 1.0

    # regularization
    lambda_l1: float = 1e-4
    lambda_l2: float = 1e-4
    lambda_var: float = 0.05     # anti-collapse: штраф за низкую дисперсию z
    min_batch_std: float = 0.10  # целевая нижняя граница std для z

    # multi-center Deep SVDD
    num_centers: int = 8
    score_mode: str = 'min'   # min | lse

    # center update
    center_momentum: float = 0.9

    # dataloader
    num_workers: int = 4
    prefetch_factor: int = 2
    persistent_workers: bool = True

cfg = OneClassConfig()
cfg


## Ожидаемые данные

- `train_pairs`: только позитивные пары (`y=1`).
- `val_pairs`: только позитивные пары (`y=1`) для early stopping / мониторинга one-class loss.
- `test_pairs`: смешанные пары (`y in {0,1}`) для подбора порога и финальной оценки.

Требуемые поля: `sku1, sku2, sku1_path, sku2_path, y`.


In [ ]:
# Пример загрузки (подставьте ваши пути)
# train_pairs = pd.read_csv('train_pairs_positive.csv')
# val_pairs   = pd.read_csv('val_pairs_positive.csv')
# test_pairs  = pd.read_csv('test_pairs_labeled.csv')

required = ['train_pairs', 'val_pairs', 'test_pairs']
for v in required:
    if v not in globals():
        raise RuntimeError(f'Variable `{v}` is not defined.')

for name, df in [('train', train_pairs), ('val', val_pairs), ('test', test_pairs)]:
    miss = {'sku1','sku2','sku1_path','sku2_path','y'} - set(df.columns)
    if miss:
        raise ValueError(f'{name} missing columns: {miss}')

assert (train_pairs['y'] == 1).all(), 'train_pairs must contain only y=1'
assert (val_pairs['y'] == 1).all(), 'val_pairs must contain only y=1'
print('Shapes:', train_pairs.shape, val_pairs.shape, test_pairs.shape)


In [ ]:
from concurrent.futures import ThreadPoolExecutor

def build_image_cache(paths, target_size: int = 224, verbose: bool = True, max_workers: int = 8):
    unique_paths = sorted(set(map(str, paths)))

    def _read_one(path: str):
        img_bgr = cv2.imread(path, cv2.IMREAD_COLOR)
        if img_bgr is None:
            raise FileNotFoundError(f'Cannot read image while caching: {path}')
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        if target_size is not None:
            img_rgb = cv2.resize(img_rgb, (target_size, target_size), interpolation=cv2.INTER_AREA)
        return path, img_rgb

    cache = {}
    if verbose:
        print(f'Caching {len(unique_paths)} unique images with {max_workers} workers...')

    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        it = ex.map(_read_one, unique_paths)
        if verbose:
            it = tqdm(it, total=len(unique_paths), desc='cache')
        for p, img in it:
            cache[p] = img
    return cache

all_paths = pd.concat([
    train_pairs[['sku1_path','sku2_path']].stack(),
    val_pairs[['sku1_path','sku2_path']].stack(),
    test_pairs[['sku1_path','sku2_path']].stack(),
], axis=0).astype(str).tolist()

IMAGE_CACHE = build_image_cache(all_paths, target_size=224, verbose=True, max_workers=16)
print('Cache size:', len(IMAGE_CACHE))


In [ ]:
class PairImageDataset(Dataset):
    def __init__(self, pair_df: pd.DataFrame, image_cache: Dict[str, np.ndarray]):
        self.df = pair_df.reset_index(drop=True)
        self.image_cache = image_cache

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        p1, p2 = str(row['sku1_path']), str(row['sku2_path'])
        if p1 not in self.image_cache or p2 not in self.image_cache:
            raise KeyError(f'Path not in cache: {p1}, {p2}')
        return self.image_cache[p1], self.image_cache[p2], np.float32(row['y'])


def make_pair_collate(processor: CLIPProcessor):
    def collate_fn(batch):
        imgs1, imgs2, ys = zip(*batch)
        px1 = processor(images=list(imgs1), return_tensors='pt')['pixel_values']
        px2 = processor(images=list(imgs2), return_tensors='pt')['pixel_values']
        ys = torch.tensor(ys, dtype=torch.float32)
        return px1, px2, ys
    return collate_fn

processor = CLIPProcessor.from_pretrained(cfg.hf_model_name)
collate = make_pair_collate(processor)

train_loader = DataLoader(PairImageDataset(train_pairs, IMAGE_CACHE), batch_size=cfg.batch_size, shuffle=True,
                          num_workers=cfg.num_workers, pin_memory=True, collate_fn=collate,
                          prefetch_factor=cfg.prefetch_factor, persistent_workers=cfg.persistent_workers)
val_loader = DataLoader(PairImageDataset(val_pairs, IMAGE_CACHE), batch_size=cfg.batch_size, shuffle=False,
                        num_workers=cfg.num_workers, pin_memory=True, collate_fn=collate,
                        prefetch_factor=cfg.prefetch_factor, persistent_workers=cfg.persistent_workers)
test_loader = DataLoader(PairImageDataset(test_pairs, IMAGE_CACHE), batch_size=cfg.batch_size, shuffle=False,
                         num_workers=cfg.num_workers, pin_memory=True, collate_fn=collate,
                         prefetch_factor=cfg.prefetch_factor, persistent_workers=cfg.persistent_workers)


In [ ]:
class SCENetOneClass(nn.Module):
    def __init__(self, clip_model_name: str, num_conditions: int, hidden_dim: int, dropout: float, svdd_dim: int):
        super().__init__()
        self.clip = CLIPModel.from_pretrained(clip_model_name)

        if hasattr(self.clip, 'visual_projection') and self.clip.visual_projection is not None:
            d = self.clip.visual_projection.out_features
        else:
            d = self.clip.config.projection_dim
        self.emb_dim = d

        self.condition_masks = nn.Parameter(torch.randn(num_conditions, d) * 0.02)
        self.weight_branch = nn.Sequential(
            nn.Linear(2 * d, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_conditions)
        )
        self.pair_projector = nn.Sequential(
            nn.Linear(2 * d, hidden_dim), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden_dim, svdd_dim)
        )

    def clip_embed(self, pixel_values):
        v = self.clip.get_image_features(pixel_values=pixel_values)
        return F.normalize(v, dim=-1)

    def sce_pair_embed(self, v1, v2):
        w = torch.softmax(self.weight_branch(torch.cat([v1, v2], dim=-1)), dim=-1)  # [B, M]
        m = self.condition_masks.unsqueeze(0)  # [1, M, D]
        e1 = (w.unsqueeze(-1) * (v1.unsqueeze(1) * m)).sum(dim=1)
        e2 = (w.unsqueeze(-1) * (v2.unsqueeze(1) * m)).sum(dim=1)
        pair = torch.cat([e1, e2], dim=-1)
        z = self.pair_projector(pair)
        return z, e1, e2, w

    def forward(self, px1, px2):
        v1 = self.clip_embed(px1)
        v2 = self.clip_embed(px2)
        return self.sce_pair_embed(v1, v2)

model = SCENetOneClass(
    clip_model_name=cfg.hf_model_name,
    num_conditions=cfg.num_conditions,
    hidden_dim=cfg.condition_hidden,
    dropout=cfg.dropout,
    svdd_dim=cfg.svdd_dim,
).to(device)

if cfg.freeze_backbone:
    for p in model.clip.parameters():
        p.requires_grad = False


In [ ]:
@torch.no_grad()
def collect_train_z(model, loader):
    model.eval()
    zs = []
    for px1, px2, _ in tqdm(loader, leave=False, desc='collect z'):
        z, _, _, _ = model(px1.to(device), px2.to(device))
        zs.append(F.normalize(z, dim=1))
    return torch.cat(zs, dim=0)

@torch.no_grad()
def init_centers_kmeans(model, loader, k=8, iters=15):
    z_all = collect_train_z(model, loader)
    n = z_all.size(0)
    idx = torch.randperm(n, device=z_all.device)[:k]
    centers = z_all[idx].clone()

    for _ in range(iters):
        d2 = torch.cdist(z_all, centers, p=2).pow(2)
        assign = d2.argmin(dim=1)
        new_centers = []
        for j in range(k):
            mask = assign == j
            if mask.any():
                cj = z_all[mask].mean(dim=0)
            else:
                cj = z_all[torch.randint(0, n, (1,), device=z_all.device)].squeeze(0)
            new_centers.append(F.normalize(cj.unsqueeze(0), dim=1).squeeze(0))
        centers = torch.stack(new_centers, dim=0)
    return centers

@torch.no_grad()
def update_centers(model, loader, centers_old, momentum=0.9):
    z_all = collect_train_z(model, loader)
    d2 = torch.cdist(z_all, centers_old, p=2).pow(2)
    assign = d2.argmin(dim=1)
    new_centers = []
    k = centers_old.size(0)
    for j in range(k):
        mask = assign == j
        if mask.any():
            cj = z_all[mask].mean(dim=0)
            cj = momentum * centers_old[j] + (1 - momentum) * cj
        else:
            cj = centers_old[j]
        new_centers.append(F.normalize(cj.unsqueeze(0), dim=1).squeeze(0))
    return torch.stack(new_centers, dim=0)

def svdd_score_from_centers(z, centers, mode='min'):
    d2 = torch.cdist(z, centers, p=2).pow(2)
    if mode == 'min':
        return d2.min(dim=1).values
    if mode == 'lse':
        return -torch.logsumexp(-d2, dim=1)
    raise ValueError(f'Unknown mode: {mode}')

def deep_svdd_loss(z, centers, e1, e2, masks, lambda_l1=1e-4, lambda_l2=1e-4, lambda_var=0.05, min_batch_std=0.1, score_mode='min'):
    z = F.normalize(z, dim=1)
    centers = F.normalize(centers, dim=1)
    dist = svdd_score_from_centers(z, centers, mode=score_mode)
    svdd = dist.mean()
    l1 = masks.abs().mean()
    l2 = 0.5 * (e1.pow(2).mean() + e2.pow(2).mean())
    batch_std = z.std(dim=0).mean()
    var_pen = F.relu(min_batch_std - batch_std)
    total = svdd + lambda_l1 * l1 + lambda_l2 * l2 + lambda_var * var_pen
    return total, {'loss': total.item(), 'svdd': svdd.item(), 'l1': l1.item(), 'l2': l2.item(), 'batch_std': batch_std.item(), 'var_pen': var_pen.item()}

optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scaler = torch.cuda.amp.GradScaler(enabled=(cfg.use_amp and torch.cuda.is_available()))

centers = init_centers_kmeans(model, train_loader, k=cfg.num_centers).to(device)
print('centers:', centers.shape)


In [ ]:
def run_epoch_oneclass(model, loader, centers, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    rows = []
    for px1, px2, _ in tqdm(loader, leave=False):
        px1 = px1.to(device, non_blocking=True)
        px2 = px2.to(device, non_blocking=True)
        with torch.set_grad_enabled(is_train):
            with torch.cuda.amp.autocast(enabled=(cfg.use_amp and torch.cuda.is_available())):
                z, e1, e2, _ = model(px1, px2)
                loss, st = deep_svdd_loss(
                    z, centers, e1, e2, model.condition_masks,
                    cfg.lambda_l1, cfg.lambda_l2, cfg.lambda_var, cfg.min_batch_std, cfg.score_mode
                )
            if is_train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.clip_grad_norm)
                scaler.step(optimizer)
                scaler.update()
        rows.append(st)
    return pd.DataFrame(rows).mean().to_dict()

history = []
best_val = float('inf')
best_state = None
for epoch in range(cfg.epochs):
    tr = run_epoch_oneclass(model, train_loader, centers, optimizer=optimizer)
    va = run_epoch_oneclass(model, val_loader, centers, optimizer=None)
    centers = update_centers(model, train_loader, centers, momentum=cfg.center_momentum).detach()

    row = {'epoch': epoch+1,
           'train_loss': tr['loss'], 'train_svdd': tr['svdd'], 'train_l1': tr['l1'], 'train_l2': tr['l2'],
           'train_batch_std': tr.get('batch_std', float('nan')),
           'val_loss': va['loss'],   'val_svdd': va['svdd'],   'val_l1': va['l1'],   'val_l2': va['l2'],
           'val_batch_std': va.get('batch_std', float('nan'))}
    history.append(row)

    if va['loss'] < best_val:
        best_val = va['loss']
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    print(row)

hist_df = pd.DataFrame(history)
hist_df


In [ ]:
if best_state is not None:
    model.load_state_dict(best_state)

@torch.no_grad()
def predict_scores(model, loader, centers):
    model.eval()
    scores, ys = [], []
    centers = F.normalize(centers, dim=1)
    for px1, px2, y in tqdm(loader, leave=False):
        z, _, _, _ = model(px1.to(device), px2.to(device))
        z = F.normalize(z, dim=1)
        s = svdd_score_from_centers(z, centers, mode=cfg.score_mode)
        scores.extend(s.cpu().numpy().tolist())
        ys.extend(y.numpy().tolist())
    return np.array(scores), np.array(ys).astype(int)

test_scores, test_y = predict_scores(model, test_loader, centers)
print('score std:', float(test_scores.std()), 'score min/max:', float(test_scores.min()), float(test_scores.max()))
# anomaly = bad pair = y==0, поэтому инвертируем label для PR/F1 по аномалии
anom_true = (test_y == 0).astype(int)

prec, rec, thr = precision_recall_curve(anom_true, test_scores)
f1 = 2 * prec * rec / (prec + rec + 1e-12)
best_i = int(np.nanargmax(f1))
best_thr = thr[max(0, best_i-1)] if len(thr) > 0 else np.median(test_scores)
pred_anom = (test_scores >= best_thr).astype(int)

print('ROC-AUC:', roc_auc_score(anom_true, test_scores))
print('PR-AUC :', average_precision_score(anom_true, test_scores))
print('best_thr:', best_thr, 'best_F1:', f1_score(anom_true, pred_anom))


## Подробно: алгоритм Deep SVDD и его перенос на SCE-Net compatibility

### 1) Идея Deep SVDD из статьи

Deep SVDD (Ruff et al.) решает one-class задачу так: обучает нейросеть \(\phi(x;	heta)\), которая отображает объекты в латентное пространство, и **сжимает нормальные объекты** внутрь компактной области около центра \(c\).

Базовая (soft-boundary/one-class) интуиция:
- если объект нормальный, его embedding должен быть близко к центру;
- если объект аномальный, его embedding окажется далеко от центра.

Практически чаще используют objective типа:
\[
\mathcal{L}_{SVDD}=rac{1}{N}\sum_i ||\phi(x_i;	heta)-c||_2^2
\]
с регуляризациями и аккуратной инициализацией \(c\) по train-данным.

Почему это работает:
- модель не учится "угадывать негативы" (которые могут быть шумными/синтетическими);
- вместо этого она учится геометрии "нормы" (good compatibility manifold).

### 2) Что является объектом x в нашей задаче

В классическом Deep SVDD объект \(x\) — одиночный пример. У нас объектом является **пара товаров** `(sku1, sku2)`.

Поэтому строим pair-encoder:
1. CLIP извлекает визуальные признаки `v1`, `v2`;
2. SCE-Net condition masks + weight branch строят **pair-conditioned** представления `e1`, `e2`;
3. из `[e1, e2]` projector выдаёт итоговый вектор пары `z = \phi(pair)`.

Дальше SVDD применяется уже к `z`.

### 3) Соответствие компонент SCE-Net ↔ Deep SVDD

- `CLIPModel`: базовое визуальное пространство.
- `condition_masks` (`M x D`): подпространства совместимости (условия похожести/сочетаемости).
- `weight_branch`: для каждой пары выдаёт веса условий `w`, т.е. выбирает, какие "углы" важны именно для этой пары.
- `pair_projector`: получает pair-conditioned признаки и делает компактный one-class embedding `z`.
- `center c`: эталон центра нормальных пар (оценка по train positives).

Итоговый score на инференсе:
\[
score(pair)=||z-c||_2^2
\]
больше score → более вероятная аномалия (bad pair).

### 4) Почему это особенно уместно при проблеме "переусложнённых негативов"

Если негативы синтетические и частично неверные, бинарная BCE-модель может:
- заучить артефакты генерации негативов,
- penalize нормальные базовые сочетания (white tee + jeans).

Deep SVDD на positives избегает прямой зависимости от качества negative sampling и смещает задачу к "насколько пара похожа на распределение реальных good looks".

### 5) Практический train/infer протокол для вашей постановки

1. **Train/Val**: только `y=1` пары (строгая фильтрация качества).
2. Инициализировать `c` средним embedding на train.
3. Обучать сеть минимизировать distance-to-center + регуляризацию масок/эмбеддингов.
4. **Test**: считать `score = ||z-c||^2` для mixed (`y=0/1`).
5. Подобрать threshold по PR/F1 (для класса "аномалия" = `y=0`).
6. Зафиксировать threshold и применять в production.

### 6) Важные нюансы из статьи и практики

- Нельзя допускать вырожденного решения (все выходы в одну точку из-за плохой параметризации) — поэтому важны архитектурные ограничения и регуляризация.
- Центр `c` обычно **фиксируют** после инициализации (как в этом ноутбуке).
- Нормализация признаков (CLIP L2 norm) стабилизирует обучение.
- Стоит делать мониторинг распределений train/val distances; сильный дрейф = проблема domain shift.

### 7) Интерпретация выхода для бизнеса

- Низкий score: пара типична для стилистически "нормальных" образов из эталонного датасета.
- Высокий score: пара вне изученного manifold (кандидат в bad / low-compatibility).
- Порог можно делать разным под бизнес-режим:
  - высокий recall bad (строже фильтр),
  - высокий precision bad (меньше false alarms).


## Что именно переиспользовано из baseline

1. **CLIP embeddings и загрузка через `CLIPModel/CLIPProcessor`** сохранены полностью.
2. **SCE-Net идея** (`condition_masks`, `weight_branch`, pair-conditioned embedding) сохранена.
3. **RAM cache изображений** (`build_image_cache`) сохранён.
4. **Dataset/Collate/DataLoader пайплайн** сохранён и адаптирован под one-class.
5. Изменена только **целевая функция**: BCE → Deep SVDD distance-to-center.
